# Phase 3 — Modeling

This notebook trains validation-only baseline models.

The test set remains locked. It will be evaluated exactly once after the final
primary model is selected.


In [10]:
import json
import joblib
import numpy as np
import pandas as pd

from pathlib import Path
from datetime import datetime

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)


In [11]:
RANDOM_STATE = 42

DATA_DIR = Path("../data/processed")
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)


In [12]:
def clean_loaded_frame(df):
    df = df.copy()

    unnamed_cols = [col for col in df.columns if str(col).startswith("Unnamed:")]
    if unnamed_cols:
        df = df.drop(columns=unnamed_cols)

    return df


def load_target(path):
    y = pd.read_csv(path)
    y = clean_loaded_frame(y)

    if y.shape[1] != 1:
        raise ValueError(f"{path} should contain exactly one target column.")

    return y.iloc[:, 0].astype(int)


In [13]:
X_train = clean_loaded_frame(pd.read_csv(DATA_DIR / "X_train.csv"))
y_train = load_target(DATA_DIR / "y_train.csv")

X_val = clean_loaded_frame(pd.read_csv(DATA_DIR / "X_val.csv"))
y_val = load_target(DATA_DIR / "y_val.csv")

print("Loaded:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:  ", X_val.shape)
print("y_val:  ", y_val.shape)


Loaded:
X_train: (413378, 905)
y_train: (413378,)
X_val:   (88581, 905)
y_val:   (88581,)


In [14]:
def validate_model_matrix(X, y, name):
    if not isinstance(X, pd.DataFrame):
        raise TypeError(f"{name}: X must be a pandas DataFrame.")

    if X.columns.duplicated().any():
        dupes = X.columns[X.columns.duplicated()].tolist()
        raise ValueError(f"{name}: duplicate columns found: {dupes[:10]}")

    non_numeric = X.select_dtypes(exclude="number").columns.tolist()
    if non_numeric:
        raise TypeError(f"{name}: non-numeric columns remain: {non_numeric[:20]}")

    if X.isna().any().any():
        raise ValueError(f"{name}: X contains NaN values.")

    if len(X) != len(y):
        raise ValueError(f"{name}: X and y length mismatch.")

    print(
        f"{name}: {X.shape[0]:,} rows | "
        f"{X.shape[1]:,} features | "
        f"fraud rate={y.mean():.4f}"
    )


validate_model_matrix(X_train, y_train, "train")
validate_model_matrix(X_val, y_val, "validation")


train: 413,378 rows | 905 features | fraud rate=0.0352
validation: 88,581 rows | 905 features | fraud rate=0.0343


In [15]:
baseline_auc_pr = y_val.mean()

print(f"Validation fraud rate baseline AUC-PR: {baseline_auc_pr:.4f}")


Validation fraud rate baseline AUC-PR: 0.0343


## Logistic Regression Baseline

This is a scalable logistic regression baseline using stochastic gradient
descent.

It is equivalent in spirit to Logistic Regression with log loss, but much faster
on this large processed fraud matrix than `LogisticRegression(solver="saga")`.

The model is evaluated on validation only.


In [29]:
logreg_baseline = Pipeline([
    (
        "scaler",
        StandardScaler(),
    ),
    (
        "model",
        SGDClassifier(
            loss="log_loss",
            penalty="l2",
            alpha=1e-4,
            class_weight="balanced",
            max_iter=1000,
            tol=1e-3,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=5,
            random_state=RANDOM_STATE,
        ),
    ),
])


In [30]:
logreg_baseline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"loss loss: {'hinge', 'log_loss', 'modified_huber', 'squared_hinge', 'perceptron', 'squared_error', 'huber', 'epsilon_insensitive', 'squared_epsilon_insensitive'}, default='hinge'The loss function to be used.- 'hinge' gives a linear SVM.- 'log_loss' gives logistic regression, a probabilistic classifier.- 'modified_huber' is another smooth loss that brings tolerance to outliers as well as probability estimates.- 'squared_hinge' is like hinge but is quadratically penalized.- 'perceptron' is the linear loss used by the perceptron algorithm.- The other losses, 'squared_error', 'huber', 'epsilon_insensitive' and 'squared_epsilon_insensitive' are designed for regression but can be useful in classification as well; see :class:`~sklearn.linear_model.SGDRegressor` for a description.More details about the losses formulas can be found in the :ref:`User Guide` and you can find a visualisation of the lossfunctions in:ref:`sphx_glr_auto_examples_linear_model_plot_sgd_loss_functions.py`.",'log_loss'
,"penalty penalty: {'l2', 'l1', 'elasticnet', None}, default='l2'The penalty (aka regularization term) to be used. Defaults to 'l2'which is the standard regularizer for linear SVM models. 'l1' and'elasticnet' might bring sparsity to the model (feature selection)not achievable with 'l2'. No penalty is added when set to `None`.You can see a visualisation of the penalties in:ref:`sphx_glr_auto_examples_linear_model_plot_sgd_penalties.py`.",'l2'
,"alpha alpha: float, default=0.0001Constant that multiplies the regu

In [31]:
val_proba = logreg_baseline.predict_proba(X_val)[:, 1]

val_auc_pr = average_precision_score(y_val, val_proba)
val_roc_auc = roc_auc_score(y_val, val_proba)

print(f"Validation AUC-PR:  {val_auc_pr:.4f}")
print(f"Validation ROC-AUC: {val_roc_auc:.4f}")
print(f"Baseline AUC-PR:    {baseline_auc_pr:.4f}")
print(f"Lift over baseline: {val_auc_pr / baseline_auc_pr:.2f}x")


Validation AUC-PR:  0.2671
Validation ROC-AUC: 0.7472
Baseline AUC-PR:    0.0343
Lift over baseline: 7.78x


In [32]:
precision, recall, thresholds = precision_recall_curve(y_val, val_proba)

threshold_df = pd.DataFrame({
    "threshold": thresholds,
    "precision": precision[:-1],
    "recall": recall[:-1],
})

threshold_df["f1"] = (
    2 * threshold_df["precision"] * threshold_df["recall"]
    / (threshold_df["precision"] + threshold_df["recall"] + 1e-12)
)

threshold_df.sort_values("f1", ascending=False).head(20)


,threshold,precision,recall,f1
86421,1.0,0.425581,0.300789,0.352465
86420,1.0,0.425384,0.300789,0.352397
86425,1.0,0.425909,0.300460,0.352352
86419,1.0,0.425186,0.300789,0.352330
86424,1.0,0.425710,0.300460,0.352284
86418,1.0,0.424988,0.300789,0.352262
86151,1.0,0.397521,0.316239,0.352252
86202,1.0,0.402279,0.313281,0.352245
86423,1.0,0.425512,0.300460,0.352216
86417,1.0,0.424791,0.300789,0.352194


In [33]:
MIN_RECALL = 0.50

candidates = threshold_df[threshold_df["recall"] >= MIN_RECALL].copy()

if len(candidates) == 0:
    raise ValueError(f"No threshold reaches recall >= {MIN_RECALL}")

selected = candidates.sort_values("precision", ascending=False).iloc[0]

best_threshold = selected["threshold"]

print(f"Selected threshold: {best_threshold:.8f}")
print(f"Precision:          {selected['precision']:.4f}")
print(f"Recall:             {selected['recall']:.4f}")
print(f"F1:                 {selected['f1']:.4f}")


Selected threshold: 0.99990631
Precision:          0.1400
Recall:             0.5000
F1:                 0.2187


In [34]:
val_pred = (val_proba >= best_threshold).astype(int)

val_precision = precision_score(y_val, val_pred, zero_division=0)
val_recall = recall_score(y_val, val_pred, zero_division=0)
val_f1 = f1_score(y_val, val_pred, zero_division=0)

val_summary = pd.DataFrame([
    {
        "model": "logistic_regression_sgd_baseline",
        "split": "validation",
        "auc_pr": val_auc_pr,
        "roc_auc": val_roc_auc,
        "threshold": best_threshold,
        "precision": val_precision,
        "recall": val_recall,
        "f1": val_f1,
    }
])

val_summary


,model,split,auc_pr,roc_auc,threshold,precision,recall,f1
0,logistic_regression_sgd_baseline,validation,0.267056,0.747158,0.999906,0.139952,0.5,0.218692


In [35]:
print("Validation confusion matrix")
print(confusion_matrix(y_val, val_pred))
print()
print(classification_report(y_val, val_pred, digits=4))


Validation confusion matrix
[[76192  9347]
 [ 1521  1521]]

              precision    recall  f1-score   support

           0     0.9804    0.8907    0.9334     85539
           1     0.1400    0.5000    0.2187      3042

    accuracy                         0.8773     88581
   macro avg     0.5602    0.6954    0.5761     88581
weighted avg     0.9516    0.8773    0.9089     88581



In [36]:
coef_df = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": logreg_baseline.named_steps["model"].coef_[0],
})

coef_df["abs_coefficient"] = coef_df["coefficient"].abs()

coef_df.sort_values("abs_coefficient", ascending=False).head(30)


,feature,coefficient,abs_coefficient
23,C14,-8.027918,8.027918
12,C3,-7.958621,7.958621
611,vgrp_76__V267_value,-6.188635,6.188635
11,C2,5.609126,5.609126
482,vgrp_74__V207_value,-5.495433,5.495433
443,vgrp_74__V168_value,-4.561185,4.561185
20,C11,4.465461,4.465461
675,vgrp_84__V147_value,4.226960,4.226960
487,vgrp_74__V212_value,-3.916664,3.916664
612,vgrp_76__V268_value,-3.903353,3.903353


In [37]:
artifact = {
    "model": logreg_baseline,
    "threshold": float(best_threshold),
    "feature_names": list(X_train.columns),
    "metadata": {
        "model_name": "logistic_regression_sgd_baseline",
        "created_at": datetime.now().isoformat(),
        "primary_metric": "validation_auc_pr",
        "validation_auc_pr": float(val_auc_pr),
        "validation_roc_auc": float(val_roc_auc),
        "validation_precision": float(val_precision),
        "validation_recall": float(val_recall),
        "validation_f1": float(val_f1),
        "baseline_auc_pr": float(baseline_auc_pr),
        "lift_over_baseline": float(val_auc_pr / baseline_auc_pr),
        "n_train_rows": int(len(X_train)),
        "n_validation_rows": int(len(X_val)),
        "n_features": int(X_train.shape[1]),
        "model_type": "SGDClassifier",
        "loss": "log_loss",
        "penalty": "l2",
        "class_weight": "balanced",
        "test_set_status": "locked_not_evaluated",
    },
}

joblib.dump(artifact, MODEL_DIR / "logreg_baseline.joblib")

with open(MODEL_DIR / "logreg_baseline_metrics.json", "w") as f:
    json.dump(artifact["metadata"], f, indent=2)

print("Saved:")
print(MODEL_DIR / "logreg_baseline.joblib")
print(MODEL_DIR / "logreg_baseline_metrics.json")


Saved:
../models/logreg_baseline.joblib
../models/logreg_baseline_metrics.json


## Baseline Notes

The Logistic Regression baseline is validation-only.

The test set is intentionally not loaded or evaluated in this notebook section.
Per the project specification, test performance will be reported exactly once
after the primary model is selected.

AUC-PR is the primary metric because the fraud class is highly imbalanced.
ROC-AUC is secondary.

The decision threshold is selected using validation data only.


### Logistic Regression Baseline Result

The SGD Logistic Regression baseline achieved validation AUC-PR of 0.2671,
compared with a fraud-rate baseline AUC-PR of 0.0343.

This confirms that the processed feature matrix contains meaningful fraud
signal.

The selected validation threshold is close to 1.0. This is expected for this
SGD-based baseline because its probability outputs are not calibrated,
especially under class weighting. Therefore, the baseline is interpreted mainly
as a ranking model using AUC-PR, not as a calibrated probability model.

The test set remains locked and was not evaluated.


## Random Forest Benchmark

Random Forest is used as a non-linear tree-based benchmark before training the primary XGBoost model.

The model is evaluated on validation only. The test set remains locked.

AUC-PR is the primary metric.

In [38]:
from sklearn.ensemble import RandomForestClassifier

In [39]:
rf_benchmark = RandomForestClassifier(
    n_estimators=300,
    max_depth=18,
    min_samples_leaf=20,
    max_features="sqrt",
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1,
)


In [40]:
rf_benchmark.fit(X_train, y_train)


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   12.2s
[Parallel(n_jobs=-1)]: Done 184 tasks      | elapsed:   54.6s
[Parallel(n_jobs=-1)]: Done 300 out of 300 | elapsed:  1.5min finished


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",18
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",20
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(

In [41]:
rf_val_proba = rf_benchmark.predict_proba(X_val)[:, 1]

rf_val_auc_pr = average_precision_score(y_val, rf_val_proba)
rf_val_roc_auc = roc_auc_score(y_val, rf_val_proba)

print(f"Random Forest Validation AUC-PR:  {rf_val_auc_pr:.4f}")
print(f"Random Forest Validation ROC-AUC: {rf_val_roc_auc:.4f}")
print(f"Baseline AUC-PR:                  {baseline_auc_pr:.4f}")
print(f"Lift over baseline:               {rf_val_auc_pr / baseline_auc_pr:.2f}x")


[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.1s
[Parallel(n_jobs=8)]: Done 184 tasks      | elapsed:    0.4s


Random Forest Validation AUC-PR:  0.4925
Random Forest Validation ROC-AUC: 0.8987
Baseline AUC-PR:                  0.0343
Lift over baseline:               14.34x


[Parallel(n_jobs=8)]: Done 300 out of 300 | elapsed:    0.6s finished


In [42]:
rf_precision, rf_recall, rf_thresholds = precision_recall_curve(
    y_val,
    rf_val_proba,
)

rf_threshold_df = pd.DataFrame({
    "threshold": rf_thresholds,
    "precision": rf_precision[:-1],
    "recall": rf_recall[:-1],
})

rf_threshold_df["f1"] = (
    2 * rf_threshold_df["precision"] * rf_threshold_df["recall"]
    / (rf_threshold_df["precision"] + rf_threshold_df["recall"] + 1e-12)
)

rf_threshold_df.sort_values("f1", ascending=False).head(20)


,threshold,precision,recall,f1
86154,0.636763,0.551139,0.437541,0.487814
86153,0.636656,0.550911,0.437541,0.487724
86152,0.636572,0.550683,0.437541,0.487635
86151,0.636371,0.550455,0.437541,0.487546
86155,0.636964,0.550953,0.437212,0.487537
86159,0.637279,0.551452,0.436884,0.487528
86060,0.630065,0.539259,0.444773,0.487480
86064,0.630505,0.539721,0.444444,0.487471
86068,0.630859,0.540184,0.444116,0.487462
86150,0.636339,0.550227,0.437541,0.487457


In [43]:
MIN_RECALL = 0.50

rf_candidates = rf_threshold_df[rf_threshold_df["recall"] >= MIN_RECALL].copy()

if len(rf_candidates) == 0:
    raise ValueError(f"No RF threshold reaches recall >= {MIN_RECALL}")

rf_selected = rf_candidates.sort_values("precision", ascending=False).iloc[0]
rf_best_threshold = rf_selected["threshold"]

print(f"Selected RF threshold: {rf_best_threshold:.8f}")
print(f"Precision:             {rf_selected['precision']:.4f}")
print(f"Recall:                {rf_selected['recall']:.4f}")
print(f"F1:                    {rf_selected['f1']:.4f}")


Selected RF threshold: 0.57811182
Precision:             0.4415
Recall:                0.5000
F1:                    0.4689


In [44]:
rf_val_pred = (rf_val_proba >= rf_best_threshold).astype(int)

rf_val_precision = precision_score(y_val, rf_val_pred, zero_division=0)
rf_val_recall = recall_score(y_val, rf_val_pred, zero_division=0)
rf_val_f1 = f1_score(y_val, rf_val_pred, zero_division=0)

print("Random Forest validation confusion matrix")
print(confusion_matrix(y_val, rf_val_pred))
print()
print(classification_report(y_val, rf_val_pred, digits=4))


Random Forest validation confusion matrix
[[83615  1924]
 [ 1521  1521]]

              precision    recall  f1-score   support

           0     0.9821    0.9775    0.9798     85539
           1     0.4415    0.5000    0.4689      3042

    accuracy                         0.9611     88581
   macro avg     0.7118    0.7388    0.7244     88581
weighted avg     0.9636    0.9611    0.9623     88581



In [45]:
rf_importance_df = pd.DataFrame({
    "feature": X_train.columns,
    "importance": rf_benchmark.feature_importances_,
})

rf_importance_df.sort_values("importance", ascending=False).head(30)


,feature,importance
22,C13,0.024244
23,C14,0.020659
14,C5,0.014550
309,vgrp_15__V70_value,0.013532
287,vgrp_15__V30_value,0.013016
72,card1_amt_mean,0.012919
286,vgrp_15__V29_value,0.012638
155,vgrp_00__V294_value,0.012604
10,C1,0.012425
308,vgrp_15__V69_value,0.012242


In [46]:
rf_artifact = {
    "model": rf_benchmark,
    "threshold": float(rf_best_threshold),
    "feature_names": list(X_train.columns),
    "metadata": {
        "model_name": "random_forest_benchmark",
        "primary_metric": "validation_auc_pr",
        "validation_auc_pr": float(rf_val_auc_pr),
        "validation_roc_auc": float(rf_val_roc_auc),
        "validation_precision": float(rf_val_precision),
        "validation_recall": float(rf_val_recall),
        "validation_f1": float(rf_val_f1),
        "baseline_auc_pr": float(baseline_auc_pr),
        "lift_over_baseline": float(rf_val_auc_pr / baseline_auc_pr),
        "n_train_rows": int(len(X_train)),
        "n_validation_rows": int(len(X_val)),
        "n_features": int(X_train.shape[1]),
        "test_set_status": "locked_not_evaluated",
    },
}

joblib.dump(rf_artifact, MODEL_DIR / "random_forest_benchmark.joblib")

with open(MODEL_DIR / "random_forest_benchmark_metrics.json", "w") as f:
    json.dump(rf_artifact["metadata"], f, indent=2)

print("Saved Random Forest benchmark.")


Saved Random Forest benchmark.


### Random Forest Benchmark Result

Random Forest achieved validation AUC-PR of 0.4925, compared with 0.2671 for
the Logistic Regression baseline and 0.0343 for the fraud-rate baseline.

This confirms that the fraud signal is strongly non-linear and that tree-based
models are better suited for this feature matrix.

At the selected validation operating point, Random Forest reaches 50% recall
with 44.15% precision. The test set remains locked.


## XGBoost Primary Model

XGBoost is the primary model for this project.

It is trained on the processed feature matrix and evaluated on validation only.
The test set remains locked until the final model is selected.

In [47]:
from xgboost import XGBClassifier

In [48]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

print(f"scale_pos_weight: {scale_pos_weight:.2f}")

scale_pos_weight: 27.43


In [49]:
xgb_primary = XGBClassifier(
    n_estimators=800,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=5,
    gamma=0.0,
    reg_alpha=0.0,
    reg_lambda=2.0,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)


In [50]:
xgb_primary.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=50,
)


[0]	validation_0-aucpr:0.29443
[50]	validation_0-aucpr:0.41413
[100]	validation_0-aucpr:0.44506
[150]	validation_0-aucpr:0.46592
[200]	validation_0-aucpr:0.48000
[250]	validation_0-aucpr:0.49004
[300]	validation_0-aucpr:0.49840
[350]	validation_0-aucpr:0.50629
[400]	validation_0-aucpr:0.51208
[450]	validation_0-aucpr:0.51758
[500]	validation_0-aucpr:0.52205
[550]	validation_0-aucpr:0.52605
[600]	validation_0-aucpr:0.53105
[650]	validation_0-aucpr:0.53423
[700]	validation_0-aucpr:0.53800
[750]	validation_0-aucpr:0.54175
[799]	validation_0-aucpr:0.54390


,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.85
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_meth

In [51]:
xgb_val_proba = xgb_primary.predict_proba(X_val)[:, 1]

xgb_val_auc_pr = average_precision_score(y_val, xgb_val_proba)
xgb_val_roc_auc = roc_auc_score(y_val, xgb_val_proba)

print(f"XGBoost Validation AUC-PR:  {xgb_val_auc_pr:.4f}")
print(f"XGBoost Validation ROC-AUC: {xgb_val_roc_auc:.4f}")
print(f"Baseline AUC-PR:            {baseline_auc_pr:.4f}")
print(f"Lift over baseline:         {xgb_val_auc_pr / baseline_auc_pr:.2f}x")


XGBoost Validation AUC-PR:  0.5440
XGBoost Validation ROC-AUC: 0.9155
Baseline AUC-PR:            0.0343
Lift over baseline:         15.84x


In [52]:
xgb_precision, xgb_recall, xgb_thresholds = precision_recall_curve(
    y_val,
    xgb_val_proba,
)

xgb_threshold_df = pd.DataFrame({
    "threshold": xgb_thresholds,
    "precision": xgb_precision[:-1],
    "recall": xgb_recall[:-1],
})

xgb_threshold_df["f1"] = (
    2 * xgb_threshold_df["precision"] * xgb_threshold_df["recall"]
    / (xgb_threshold_df["precision"] + xgb_threshold_df["recall"] + 1e-12)
)

xgb_threshold_df.sort_values("f1", ascending=False).head(20)


,threshold,precision,recall,f1
85954,0.802637,0.608470,0.462853,0.525765
86144,0.819483,0.639360,0.446417,0.525745
85965,0.803519,0.610074,0.461867,0.525725
85953,0.802621,0.608207,0.462853,0.525667
85968,0.803751,0.610435,0.461538,0.525646
86143,0.819412,0.639059,0.446417,0.525644
85964,0.803378,0.609809,0.461867,0.525627
86135,0.818846,0.637600,0.447074,0.525604
85952,0.802573,0.607945,0.462853,0.525569
85971,0.804088,0.610797,0.461210,0.525567


In [53]:
MIN_RECALL = 0.50

xgb_candidates = xgb_threshold_df[xgb_threshold_df["recall"] >= MIN_RECALL].copy()

if len(xgb_candidates) == 0:
    raise ValueError(f"No XGBoost threshold reaches recall >= {MIN_RECALL}")

xgb_selected = xgb_candidates.sort_values("precision", ascending=False).iloc[0]
xgb_best_threshold = xgb_selected["threshold"]

print(f"Selected XGBoost threshold: {xgb_best_threshold:.8f}")
print(f"Precision:                  {xgb_selected['precision']:.4f}")
print(f"Recall:                     {xgb_selected['recall']:.4f}")
print(f"F1:                         {xgb_selected['f1']:.4f}")


Selected XGBoost threshold: 0.76592571
Precision:                  0.5465
Recall:                     0.5000
F1:                         0.5222


In [54]:
xgb_val_pred = (xgb_val_proba >= xgb_best_threshold).astype(int)

xgb_val_precision = precision_score(y_val, xgb_val_pred, zero_division=0)
xgb_val_recall = recall_score(y_val, xgb_val_pred, zero_division=0)
xgb_val_f1 = f1_score(y_val, xgb_val_pred, zero_division=0)

print("XGBoost validation confusion matrix")
print(confusion_matrix(y_val, xgb_val_pred))
print()
print(classification_report(y_val, xgb_val_pred, digits=4))


XGBoost validation confusion matrix
[[84277  1262]
 [ 1521  1521]]

              precision    recall  f1-score   support

           0     0.9823    0.9852    0.9838     85539
           1     0.5465    0.5000    0.5222      3042

    accuracy                         0.9686     88581
   macro avg     0.7644    0.7426    0.7530     88581
weighted avg     0.9673    0.9686    0.9679     88581



In [55]:
model_comparison = pd.DataFrame([
    {
        "model": "logistic_regression_sgd",
        "validation_auc_pr": val_auc_pr,
    },
    {
        "model": "random_forest",
        "validation_auc_pr": rf_val_auc_pr,
    },
    {
        "model": "xgboost",
        "validation_auc_pr": xgb_val_auc_pr,
    },
])

model_comparison.sort_values("validation_auc_pr", ascending=False)


,model,validation_auc_pr
2,xgboost,0.543958
1,random_forest,0.492466
0,logistic_regression_sgd,0.267056


## XGBoost Tuning

The first XGBoost model improved over both baselines but did not reach the
target validation AUC-PR of 0.60.

The test set remains locked. Tuning is performed using validation AUC-PR only.


In [17]:
from xgboost import XGBClassifier

In [18]:
xgb_tuning_results = []

scale_pos_weight_full = (y_train == 0).sum() / (y_train == 1).sum()
scale_pos_weight_sqrt = np.sqrt(scale_pos_weight_full)

xgb_candidates = [
    {
        "name": "xgb_depth4_sqrt_weight",
        "max_depth": 4,
        "learning_rate": 0.03,
        "min_child_weight": 5,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "reg_lambda": 2.0,
        "reg_alpha": 0.0,
        "scale_pos_weight": scale_pos_weight_sqrt,
    },
    {
        "name": "xgb_depth5_sqrt_weight",
        "max_depth": 5,
        "learning_rate": 0.03,
        "min_child_weight": 5,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "reg_lambda": 2.0,
        "reg_alpha": 0.0,
        "scale_pos_weight": scale_pos_weight_sqrt,
    },
    {
        "name": "xgb_depth6_sqrt_weight",
        "max_depth": 6,
        "learning_rate": 0.025,
        "min_child_weight": 8,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "reg_lambda": 3.0,
        "reg_alpha": 0.0,
        "scale_pos_weight": scale_pos_weight_sqrt,
    },
    {
        "name": "xgb_depth5_full_weight",
        "max_depth": 5,
        "learning_rate": 0.03,
        "min_child_weight": 5,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "reg_lambda": 2.0,
        "reg_alpha": 0.0,
        "scale_pos_weight": scale_pos_weight_full,
    },
    {
        "name": "xgb_depth5_no_weight",
        "max_depth": 5,
        "learning_rate": 0.03,
        "min_child_weight": 5,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "reg_lambda": 2.0,
        "reg_alpha": 0.0,
        "scale_pos_weight": 1.0,
    },
]


In [19]:
best_xgb_model = None
best_xgb_auc_pr = -np.inf
best_xgb_name = None

for candidate in xgb_candidates:
    name = candidate["name"]
    params = {k: v for k, v in candidate.items() if k != "name"}

    print(f"\nTraining {name}")

    model = XGBClassifier(
        n_estimators=1500,
        objective="binary:logistic",
        eval_metric="aucpr",
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        early_stopping_rounds=75,
        **params,
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        verbose=100,
    )

    val_proba_candidate = model.predict_proba(X_val)[:, 1]

    auc_pr_candidate = average_precision_score(y_val, val_proba_candidate)
    roc_auc_candidate = roc_auc_score(y_val, val_proba_candidate)

    result = {
        "model": name,
        "validation_auc_pr": auc_pr_candidate,
        "validation_roc_auc": roc_auc_candidate,
        "best_iteration": model.best_iteration,
        **params,
    }

    xgb_tuning_results.append(result)

    if auc_pr_candidate > best_xgb_auc_pr:
        best_xgb_auc_pr = auc_pr_candidate
        best_xgb_model = model
        best_xgb_name = name



Training xgb_depth4_sqrt_weight
[0]	validation_0-aucpr:0.30952
[100]	validation_0-aucpr:0.44036
[200]	validation_0-aucpr:0.47483
[300]	validation_0-aucpr:0.49315
[400]	validation_0-aucpr:0.50391
[500]	validation_0-aucpr:0.51364
[600]	validation_0-aucpr:0.52306
[700]	validation_0-aucpr:0.52966
[800]	validation_0-aucpr:0.53566
[900]	validation_0-aucpr:0.54062
[1000]	validation_0-aucpr:0.54370
[1100]	validation_0-aucpr:0.54689
[1200]	validation_0-aucpr:0.55096
[1300]	validation_0-aucpr:0.55478
[1400]	validation_0-aucpr:0.55628
[1499]	validation_0-aucpr:0.55938

Training xgb_depth5_sqrt_weight


Exception ignored while calling ctypes callback function <bound method DataIter._next_wrapper of <xgboost.data.SingleBatchInternalIter object at 0x13a4fbd90>>:
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/core.py", line 607, in _next_wrapper
    def _next_wrapper(self, this: None) -> int:  # pylint: disable=unused-argument
KeyboardInterrupt: 


XGBoostError: [22:15:01] /Users/runner/work/xgboost/xgboost/src/data/quantile_dmatrix.cc:179: Check failed: accumulated_rows == info.num_row_ (413378 vs. 826756) : 
Stack trace:
  [bt] (0) 1   libxgboost.dylib                    0x000000013f803dd8 dmlc::LogMessageFatal::~LogMessageFatal() + 124
  [bt] (1) 2   libxgboost.dylib                    0x000000013fa362c8 xgboost::data::cpu_impl::MakeSketches(xgboost::Context const*, xgboost::data::DataIterProxy<void (void*), int (void*)>*, xgboost::data::DMatrixProxy*, std::__1::shared_ptr<xgboost::DMatrix>, float, xgboost::common::HistogramCuts*, xgboost::BatchParam const&, xgboost::MetaInfo const&, xgboost::data::ExternalDataInfo const&, std::__1::vector<xgboost::FeatureType, std::__1::allocator<xgboost::FeatureType>>*) + 2504
  [bt] (2) 3   libxgboost.dylib                    0x000000013fa1538c xgboost::data::IterativeDMatrix::InitFromCPU(xgboost::Context const*, xgboost::BatchParam const&, xgboost::data::DataIterProxy<void (void*), int (vo

In [ ]:
xgb_tuning_df = pd.DataFrame(xgb_tuning_results)

xgb_tuning_df.sort_values(
    "validation_auc_pr",
    ascending=False,
)


,model,validation_auc_pr,validation_roc_auc,best_iteration,max_depth,learning_rate,min_child_weight,subsample,colsample_bytree,reg_lambda,reg_alpha,scale_pos_weight
2,xgb_depth6_sqrt_weight,0.608064,0.930206,1499,6,0.025,8,0.85,0.85,3.0,0.0,5.237777
4,xgb_depth5_no_weight,0.593923,0.921065,1499,5,0.030,5,0.85,0.85,2.0,0.0,1.000000
1,xgb_depth5_sqrt_weight,0.589318,0.925118,1499,5,0.030,5,0.85,0.85,2.0,0.0,5.237777
3,xgb_depth5_full_weight,0.573635,0.922841,1499,5,0.030,5,0.85,0.85,2.0,0.0,27.434310
0,xgb_depth4_sqrt_weight,0.559429,0.914000,1499,4,0.030,5,0.85,0.85,2.0,0.0,5.237777


In [ ]:
print(f"Best tuned XGBoost: {best_xgb_name}")
print(f"Best validation AUC-PR: {best_xgb_auc_pr:.4f}")


Best tuned XGBoost: xgb_depth6_sqrt_weight
Best validation AUC-PR: 0.6081


## Extended XGBoost Primary Candidate

The best tuned model reached validation AUC-PR above 0.60, but early stopping
did not trigger because the best iteration was the final boosting round.

A longer run is trained with the same hyperparameters to check whether the model
continues improving before selecting the final primary model.

The test set remains locked.

In [20]:
from xgboost import XGBClassifier

In [21]:
xgb_extended = XGBClassifier(
    n_estimators=3000,
    max_depth=6,
    learning_rate=0.025,
    min_child_weight=8,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=3.0,
    reg_alpha=0.0,
    scale_pos_weight=scale_pos_weight_sqrt,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    early_stopping_rounds=100,
)

In [22]:
xgb_extended.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=100,
)


[0]	validation_0-aucpr:0.34963
[100]	validation_0-aucpr:0.46807
[200]	validation_0-aucpr:0.50378
[300]	validation_0-aucpr:0.52206
[400]	validation_0-aucpr:0.53652
[500]	validation_0-aucpr:0.54911
[600]	validation_0-aucpr:0.55999
[700]	validation_0-aucpr:0.56704
[800]	validation_0-aucpr:0.57339
[900]	validation_0-aucpr:0.58008
[1000]	validation_0-aucpr:0.58531
[1100]	validation_0-aucpr:0.59146
[1200]	validation_0-aucpr:0.59662
[1300]	validation_0-aucpr:0.60111
[1400]	validation_0-aucpr:0.60506
[1500]	validation_0-aucpr:0.60804
[1600]	validation_0-aucpr:0.61218
[1700]	validation_0-aucpr:0.61546
[1800]	validation_0-aucpr:0.61859
[1900]	validation_0-aucpr:0.62113
[2000]	validation_0-aucpr:0.62416
[2100]	validation_0-aucpr:0.62667
[2200]	validation_0-aucpr:0.62792
[2300]	validation_0-aucpr:0.62998
[2400]	validation_0-aucpr:0.63106
[2500]	validation_0-aucpr:0.63360
[2600]	validation_0-aucpr:0.63481
[2700]	validation_0-aucpr:0.63602
[2800]	validation_0-aucpr:0.63667
[2900]	validation_0-aucpr:

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.85
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",100
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_metho

In [23]:
xgb_extended_val_proba = xgb_extended.predict_proba(X_val)[:, 1]

xgb_extended_val_auc_pr = average_precision_score(y_val, xgb_extended_val_proba)
xgb_extended_val_roc_auc = roc_auc_score(y_val, xgb_extended_val_proba)

print(f"Extended XGBoost Validation AUC-PR:  {xgb_extended_val_auc_pr:.4f}")
print(f"Extended XGBoost Validation ROC-AUC: {xgb_extended_val_roc_auc:.4f}")
print(f"Best iteration: {xgb_extended.best_iteration}")


Extended XGBoost Validation AUC-PR:  0.6391
Extended XGBoost Validation ROC-AUC: 0.9346
Best iteration: 2999


In [25]:
final_xgb_comparison = pd.DataFrame([
    {
        "model": "xgb_tuned_depth6_sqrt_weight_1500",
        "validation_auc_pr": 0.6081,
        "validation_roc_auc": 0.9302,
        "best_iteration": 1499,
    },
    {
        "model": "xgb_extended_depth6_sqrt_weight_3000",
        "validation_auc_pr": xgb_extended_val_auc_pr,
        "validation_roc_auc": xgb_extended_val_roc_auc,
        "best_iteration": xgb_extended.best_iteration,
    },
])
final_xgb_comparison.sort_values("validation_auc_pr", ascending=False)


,model,validation_auc_pr,validation_roc_auc,best_iteration
1,xgb_extended_depth6_sqrt_weight_3000,0.639054,0.934598,2999
0,xgb_tuned_depth6_sqrt_weight_1500,0.608100,0.930200,1499


In [26]:
from sklearn.metrics import average_precision_score

xgb_train_proba = xgb_extended.predict_proba(X_train)[:, 1]
xgb_train_auc_pr = average_precision_score(y_train, xgb_train_proba)

print(f"Train AUC-PR: {xgb_train_auc_pr:.4f}")
print(f"Val AUC-PR:   {xgb_extended_val_auc_pr:.4f}")
print(f"Gap:          {xgb_train_auc_pr - xgb_extended_val_auc_pr:.4f}")

Train AUC-PR: 0.9439
Val AUC-PR:   0.6391
Gap:          0.3048


## Regularized XGBoost Candidate

The 3000-tree model exceeds the validation AUC-PR target but shows a large
train-validation gap. The next candidate reduces overfitting by using stronger
regularization, lower column sampling, and more conservative tree growth.

The test set remains locked.


In [27]:
xgb_regularized = XGBClassifier(
    n_estimators=3000,
    max_depth=4,
    learning_rate=0.025,
    min_child_weight=20,
    subsample=0.80,
    colsample_bytree=0.70,
    reg_lambda=8.0,
    reg_alpha=1.0,
    gamma=1.0,
    scale_pos_weight=scale_pos_weight_sqrt,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    early_stopping_rounds=150,
)

In [28]:
xgb_regularized.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=100,
)


[0]	validation_0-aucpr:0.31204
[100]	validation_0-aucpr:0.43232
[200]	validation_0-aucpr:0.46332
[300]	validation_0-aucpr:0.48125
[400]	validation_0-aucpr:0.49294
[500]	validation_0-aucpr:0.50136
[600]	validation_0-aucpr:0.51017
[700]	validation_0-aucpr:0.51738
[800]	validation_0-aucpr:0.52442
[900]	validation_0-aucpr:0.53002
[1000]	validation_0-aucpr:0.53486
[1100]	validation_0-aucpr:0.53767
[1200]	validation_0-aucpr:0.54186
[1300]	validation_0-aucpr:0.54497
[1400]	validation_0-aucpr:0.54784
[1500]	validation_0-aucpr:0.55125
[1600]	validation_0-aucpr:0.55478
[1700]	validation_0-aucpr:0.55813
[1800]	validation_0-aucpr:0.56034
[1900]	validation_0-aucpr:0.56309
[2000]	validation_0-aucpr:0.56505
[2100]	validation_0-aucpr:0.56781
[2200]	validation_0-aucpr:0.56969
[2300]	validation_0-aucpr:0.57141
[2400]	validation_0-aucpr:0.57373
[2500]	validation_0-aucpr:0.57631
[2600]	validation_0-aucpr:0.57775
[2700]	validation_0-aucpr:0.57942
[2800]	validation_0-aucpr:0.58099
[2900]	validation_0-aucpr:

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.7
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",150
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method

In [29]:
xgb_regularized_train_proba = xgb_regularized.predict_proba(X_train)[:, 1]
xgb_regularized_val_proba = xgb_regularized.predict_proba(X_val)[:, 1]

xgb_regularized_train_auc_pr = average_precision_score(
    y_train,
    xgb_regularized_train_proba,
)
xgb_regularized_val_auc_pr = average_precision_score(
    y_val,
    xgb_regularized_val_proba,
)
xgb_regularized_val_roc_auc = roc_auc_score(y_val, xgb_regularized_val_proba)

print(f"Regularized XGBoost Train AUC-PR: {xgb_regularized_train_auc_pr:.4f}")
print(f"Regularized XGBoost Val AUC-PR:   {xgb_regularized_val_auc_pr:.4f}")
print(f"Regularized XGBoost Val ROC-AUC:  {xgb_regularized_val_roc_auc:.4f}")
print(f"Gap:                              {xgb_regularized_train_auc_pr - xgb_regularized_val_auc_pr:.4f}")
print(f"Best iteration:                   {xgb_regularized.best_iteration}")


Regularized XGBoost Train AUC-PR: 0.7973
Regularized XGBoost Val AUC-PR:   0.5851
Regularized XGBoost Val ROC-AUC:  0.9231
Gap:                              0.2123
Best iteration:                   2990


In [31]:
xgb_extended_train_proba = xgb_extended.predict_proba(X_train)[:, 1]
xgb_extended_train_auc_pr = average_precision_score(y_train, xgb_extended_train_proba)
overfit_comparison = pd.DataFrame([
    {
        "model": "xgb_3000_depth6",
        "train_auc_pr": xgb_extended_train_auc_pr,
        "val_auc_pr": xgb_extended_val_auc_pr,
        "gap": xgb_extended_train_auc_pr - xgb_extended_val_auc_pr,
        "best_iteration": xgb_extended.best_iteration,
    },
    {
        "model": "xgb_regularized_depth4",
        "train_auc_pr": xgb_regularized_train_auc_pr,
        "val_auc_pr": xgb_regularized_val_auc_pr,
        "gap": xgb_regularized_train_auc_pr - xgb_regularized_val_auc_pr,
        "best_iteration": xgb_regularized.best_iteration,
    },
])

overfit_comparison.sort_values("val_auc_pr", ascending=False)


,model,train_auc_pr,val_auc_pr,gap,best_iteration
0,xgb_3000_depth6,0.943880,0.639054,0.304826,2999
1,xgb_regularized_depth4,0.797336,0.585055,0.212281,2990


## Generalization-Focused XGBoost Candidate

A single middle-ground candidate is trained to reduce overfitting while keeping
validation AUC-PR above the 0.60 project target.

The test set remains locked.


In [32]:
xgb_generalized = XGBClassifier(
    n_estimators=3000,
    max_depth=5,
    learning_rate=0.025,
    min_child_weight=12,
    subsample=0.80,
    colsample_bytree=0.75,
    reg_lambda=5.0,
    reg_alpha=0.5,
    gamma=0.5,
    scale_pos_weight=scale_pos_weight_sqrt,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    early_stopping_rounds=150,
)


In [33]:
xgb_generalized.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=100,
)


[0]	validation_0-aucpr:0.32922
[100]	validation_0-aucpr:0.45109
[200]	validation_0-aucpr:0.48418
[300]	validation_0-aucpr:0.50260
[400]	validation_0-aucpr:0.51443
[500]	validation_0-aucpr:0.52467
[600]	validation_0-aucpr:0.53394
[700]	validation_0-aucpr:0.54083
[800]	validation_0-aucpr:0.54786
[900]	validation_0-aucpr:0.55373
[1000]	validation_0-aucpr:0.55785
[1100]	validation_0-aucpr:0.56226
[1200]	validation_0-aucpr:0.56717
[1300]	validation_0-aucpr:0.57124
[1400]	validation_0-aucpr:0.57474
[1500]	validation_0-aucpr:0.57779
[1600]	validation_0-aucpr:0.58177
[1700]	validation_0-aucpr:0.58464
[1800]	validation_0-aucpr:0.58771
[1900]	validation_0-aucpr:0.58986
[2000]	validation_0-aucpr:0.59268
[2100]	validation_0-aucpr:0.59489
[2200]	validation_0-aucpr:0.59764
[2300]	validation_0-aucpr:0.59905
[2400]	validation_0-aucpr:0.60195
[2500]	validation_0-aucpr:0.60412
[2600]	validation_0-aucpr:0.60651
[2700]	validation_0-aucpr:0.60831
[2800]	validation_0-aucpr:0.61018
[2900]	validation_0-aucpr:

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.75
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",150
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_metho

In [34]:
xgb_generalized_train_proba = xgb_generalized.predict_proba(X_train)[:, 1]
xgb_generalized_val_proba = xgb_generalized.predict_proba(X_val)[:, 1]

xgb_generalized_train_auc_pr = average_precision_score(
    y_train,
    xgb_generalized_train_proba,
)
xgb_generalized_val_auc_pr = average_precision_score(
    y_val,
    xgb_generalized_val_proba,
)
xgb_generalized_val_roc_auc = roc_auc_score(y_val, xgb_generalized_val_proba)

print(f"Generalized XGBoost Train AUC-PR: {xgb_generalized_train_auc_pr:.4f}")
print(f"Generalized XGBoost Val AUC-PR:   {xgb_generalized_val_auc_pr:.4f}")
print(f"Generalized XGBoost Val ROC-AUC:  {xgb_generalized_val_roc_auc:.4f}")
print(f"Gap:                              {xgb_generalized_train_auc_pr - xgb_generalized_val_auc_pr:.4f}")
print(f"Best iteration:                   {xgb_generalized.best_iteration}")


Generalized XGBoost Train AUC-PR: 0.8767
Generalized XGBoost Val AUC-PR:   0.6130
Generalized XGBoost Val ROC-AUC:  0.9306
Gap:                              0.2637
Best iteration:                   2998


In [35]:
generalization_comparison = pd.DataFrame([
    {
        "model": "xgb_depth6_3000_best_val",
        "train_auc_pr": xgb_extended_train_auc_pr,
        "val_auc_pr": xgb_extended_val_auc_pr,
        "gap": xgb_extended_train_auc_pr - xgb_extended_val_auc_pr,
        "best_iteration": xgb_extended.best_iteration,
    },
    {
        "model": "xgb_depth4_regularized",
        "train_auc_pr": xgb_regularized_train_auc_pr,
        "val_auc_pr": xgb_regularized_val_auc_pr,
        "gap": xgb_regularized_train_auc_pr - xgb_regularized_val_auc_pr,
        "best_iteration": xgb_regularized.best_iteration,
    },
    {
        "model": "xgb_depth5_generalized",
        "train_auc_pr": xgb_generalized_train_auc_pr,
        "val_auc_pr": xgb_generalized_val_auc_pr,
        "gap": xgb_generalized_train_auc_pr - xgb_generalized_val_auc_pr,
        "best_iteration": xgb_generalized.best_iteration,
    },
])

generalization_comparison.sort_values("val_auc_pr", ascending=False)


,model,train_auc_pr,val_auc_pr,gap,best_iteration
0,xgb_depth6_3000_best_val,0.943880,0.639054,0.304826,2999
2,xgb_depth5_generalized,0.876683,0.613025,0.263658,2998
1,xgb_depth4_regularized,0.797336,0.585055,0.212281,2990


### Generalized XGBoost Result

The depth-5 generalized XGBoost model achieved validation AUC-PR of 0.6130,
exceeding the project target of 0.60.

Compared with the depth-6 extended model, it has lower validation AUC-PR but a
smaller train-validation gap. It is selected as the primary model because it
better balances performance and generalization.

The test set remains locked until final one-time evaluation.


In [36]:
primary_model = xgb_generalized
primary_model_name = "xgb_depth5_generalized"

primary_train_auc_pr = xgb_generalized_train_auc_pr
primary_val_auc_pr = xgb_generalized_val_auc_pr
primary_val_roc_auc = xgb_generalized_val_roc_auc
primary_val_proba = xgb_generalized_val_proba
primary_gap = primary_train_auc_pr - primary_val_auc_pr

print(f"Selected primary model: {primary_model_name}")
print(f"Train AUC-PR:           {primary_train_auc_pr:.4f}")
print(f"Validation AUC-PR:      {primary_val_auc_pr:.4f}")
print(f"Validation ROC-AUC:     {primary_val_roc_auc:.4f}")
print(f"Train-Val Gap:          {primary_gap:.4f}")
print(f"Best iteration:         {primary_model.best_iteration}")

Selected primary model: xgb_depth5_generalized
Train AUC-PR:           0.8767
Validation AUC-PR:      0.6130
Validation ROC-AUC:     0.9306
Train-Val Gap:          0.2637
Best iteration:         2998


In [37]:
joblib.dump({
    "model": xgb_generalized,
    "feature_names": list(X_train.columns),
    "metadata": {
        "model_name": "xgb_generalized_primary",
        "val_auc_pr": float(xgb_generalized_val_auc_pr),
        "val_roc_auc": float(xgb_generalized_val_roc_auc),
        "train_auc_pr": float(xgb_generalized_train_auc_pr),
        "gap": float(xgb_generalized_train_auc_pr - xgb_generalized_val_auc_pr),
        "best_iteration": int(xgb_generalized.best_iteration),
        "test_set_status": "locked_not_evaluated",
    },
}, MODEL_DIR / "xgb_generalized_primary.joblib")

print("Saved.")

Saved.


In [2]:
import os
os.chdir("/Users/ilyas/Desktop/ieee-fraud-detection")

import joblib
from pathlib import Path

MODEL_DIR = Path("models")
artifact = joblib.load(MODEL_DIR / "xgb_generalized_primary.joblib")
xgb_generalized = artifact["model"]
print("Loaded:", artifact["metadata"]["model_name"])
print("Val AUC-PR:", artifact["metadata"]["val_auc_pr"])

Loaded: xgb_generalized_primary
Val AUC-PR: 0.6130247568220674


In [3]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

DATA_DIR = Path("data/processed")

def clean_loaded_frame(df):
    df = df.copy()
    unnamed_cols = [col for col in df.columns if str(col).startswith("Unnamed:")]
    if unnamed_cols:
        df = df.drop(columns=unnamed_cols)
    return df

X_val = clean_loaded_frame(pd.read_csv(DATA_DIR / "X_val.csv"))
y_val = pd.read_csv(DATA_DIR / "y_val.csv").squeeze().astype(int)

val_proba = xgb_generalized.predict_proba(X_val)[:, 1]

# Build precision-recall curve
precision, recall, thresholds = precision_recall_curve(y_val, val_proba)
f1_scores = 2 * precision * recall / (precision + recall + 1e-12)

# F1-max threshold
best_f1_idx = np.argmax(f1_scores[:-1])
f1_max_threshold = thresholds[best_f1_idx]

print("--- F1-Max Threshold ---")
print(f"Threshold:  {f1_max_threshold:.6f}")
print(f"Precision:  {precision[best_f1_idx]:.4f}")
print(f"Recall:     {recall[best_f1_idx]:.4f}")
print(f"F1:         {f1_scores[best_f1_idx]:.4f}")

# Recall-constrained threshold (recall >= 0.50)
threshold_df = pd.DataFrame({
    "threshold": thresholds,
    "precision": precision[:-1],
    "recall": recall[:-1],
    "f1": f1_scores[:-1],
})

candidates = threshold_df[threshold_df["recall"] >= 0.50]
selected = candidates.sort_values("precision", ascending=False).iloc[0]
recall_constrained_threshold = selected["threshold"]

print("\n--- Recall-Constrained Threshold (recall >= 0.50) ---")
print(f"Threshold:  {recall_constrained_threshold:.6f}")
print(f"Precision:  {selected['precision']:.4f}")
print(f"Recall:     {selected['recall']:.4f}")
print(f"F1:         {selected['f1']:.4f}")

--- F1-Max Threshold ---
Threshold:  0.480814
Precision:  0.7196
Recall:     0.4987
F1:         0.5891

--- Recall-Constrained Threshold (recall >= 0.50) ---
Threshold:  0.474283
Precision:  0.7098
Recall:     0.5000
F1:         0.5867


In [4]:
# Business cost simulation
# Cost = (FN × $250) + (FP × $15)
# FN = missed fraud — criminal gets through
# FP = false alarm — legitimate customer blocked

COST_FN = 250
COST_FP = 15

total_fraud = y_val.sum()
total_legit = (y_val == 0).sum()

cost_df = threshold_df.copy()

cost_df["TP"] = cost_df["recall"] * total_fraud
cost_df["FN"] = total_fraud - cost_df["TP"]
cost_df["FP"] = cost_df.apply(
    lambda row: (row["TP"] / row["precision"] - row["TP"]) if row["precision"] > 0 else total_legit,
    axis=1,
)

cost_df["total_cost"] = (cost_df["FN"] * COST_FN) + (cost_df["FP"] * COST_FP)

# Baseline: no model, flag nothing — all fraud becomes FN
baseline_cost = total_fraud * COST_FN
print(f"Baseline cost (no model): ${baseline_cost:,.0f}")

# Best threshold by cost
best_cost_idx = cost_df["total_cost"].idxmin()
best_cost_row = cost_df.loc[best_cost_idx]

business_threshold = best_cost_row["threshold"]

print(f"\n--- Business Cost Threshold ---")
print(f"Threshold:    {business_threshold:.6f}")
print(f"Precision:    {best_cost_row['precision']:.4f}")
print(f"Recall:       {best_cost_row['recall']:.4f}")
print(f"F1:           {best_cost_row['f1']:.4f}")
print(f"FN:           {best_cost_row['FN']:.0f}")
print(f"FP:           {best_cost_row['FP']:.0f}")
print(f"Total cost:   ${best_cost_row['total_cost']:,.0f}")
print(f"Cost savings vs baseline: ${baseline_cost - best_cost_row['total_cost']:,.0f}")
print(f"\n--- Threshold Comparison ---")
print(f"F1-max threshold:       {f1_max_threshold:.6f} | cost: ${cost_df.loc[cost_df['threshold'].sub(f1_max_threshold).abs().idxmin(), 'total_cost']:,.0f}")
print(f"Business threshold:     {business_threshold:.6f} | cost: ${best_cost_row['total_cost']:,.0f}")

Baseline cost (no model): $760,500

--- Business Cost Threshold ---
Threshold:    0.110426
Precision:    0.2601
Recall:       0.7807
F1:           0.3902
FN:           667
FP:           6755
Total cost:   $268,075
Cost savings vs baseline: $492,425

--- Threshold Comparison ---
F1-max threshold:       0.480814 | cost: $390,115
Business threshold:     0.110426 | cost: $268,075


In [5]:
artifact["thresholds"] = {
    "f1_max": float(f1_max_threshold),
    "business_cost": float(business_threshold),
    "deployed": float(business_threshold),
}
artifact["metadata"]["business_cost"] = float(best_cost_row["total_cost"])
artifact["metadata"]["baseline_cost"] = float(baseline_cost)
artifact["metadata"]["cost_savings"] = float(baseline_cost - best_cost_row["total_cost"])

joblib.dump(artifact, MODEL_DIR / "xgb_generalized_primary.joblib")
print("Artifact updated with thresholds.")

Artifact updated with thresholds.


In [7]:
import mlflow

mlflow.set_experiment("ieee-fraud-detection")

with mlflow.start_run(run_name="xgb_generalized_primary"):
    
    # Parameters
    mlflow.log_param("model_type", "XGBClassifier")
    mlflow.log_param("max_depth", 5)
    mlflow.log_param("learning_rate", 0.025)
    mlflow.log_param("n_estimators", 3000)
    mlflow.log_param("best_iteration", 2998)
    mlflow.log_param("min_child_weight", 12)
    mlflow.log_param("subsample", 0.80)
    mlflow.log_param("colsample_bytree", 0.75)
    mlflow.log_param("reg_lambda", 5.0)
    mlflow.log_param("reg_alpha", 0.5)
    mlflow.log_param("gamma", 0.5)
    mlflow.log_param("scale_pos_weight", "sqrt")
    
    # Metrics
    mlflow.log_metric("train_auc_pr", 0.8767)
    mlflow.log_metric("val_auc_pr", 0.6130)
    mlflow.log_metric("val_roc_auc", 0.9306)
    mlflow.log_metric("train_val_gap", 0.2637)
    mlflow.log_metric("f1_max_threshold", f1_max_threshold)
    mlflow.log_metric("business_threshold", business_threshold)
    mlflow.log_metric("business_cost", best_cost_row["total_cost"])
    mlflow.log_metric("baseline_cost", baseline_cost)
    mlflow.log_metric("cost_savings", baseline_cost - best_cost_row["total_cost"])
    
    # Artifact
    mlflow.log_artifact("models/xgb_generalized_primary.joblib")
    
    print("Run logged successfully.")

2026/05/11 21:19:46 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/11 21:19:46 INFO mlflow.store.db.utils: Updating database tables
2026/05/11 21:19:47 INFO mlflow.tracking.fluent: Experiment with name 'ieee-fraud-detection' does not exist. Creating a new experiment.


Run logged successfully.
